In [ ]:
import dask
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

from srm.qa_flags import (
    ATTRS_TIME_INVARIANT,
    ATTRS_TIME_VARYING,
    FLAG_LIST_TIME_INVARIANT,
    FLAG_LIST_TIME_VARYING,
    combine_intermediate_flags,
    get_intermediate_flags,
    parse_tag,
    write_final_qa_flags,
)

In [ ]:
import os

import frisky
from distributed import Client

os.environ["FRISKY_SUMMARY"] = "off"
os.environ["FRISKY_DEATH_DUMP_DIR"] = ""


client = frisky.hijack(Client())
client

# A. Define what data arrays exist to traverse

In [ ]:
# --- Run parameters --------------------------------------------------------
# This regional South-Africa-box run covers three GCMs, each written to its own icechunk store
# (same bucket/branch, named by GCM the same way srm.cache.ArtifactCache names pipeline output
# stores). Looping over GCMS -- rather than hardcoding one, as this notebook used to -- is what
# lets every leaf be compared against ITS OWN GCM's catalog and lineage instead of silently
# reusing whichever GCM happened to be hardcoded.


VARIABLES = ["tas", "tasmax", "tasmin", "pr", "rsds"]  # , "hurs"]

In [ ]:
GCMS = ["CESM2-WACCM"]

# BRANCH = "full-regional-run-issue-534"
# ROOT_DIR = "s3://carbonplan-scratch/srm/output/qa/"
# STORE_SUBSET_BOUNDS = (-38.0, -19.0, 13.0, 36.0)
# STORE_SUBSET_ID = ArtifactCache._get_subset_id(STORE_SUBSET_BOUNDS)

BRANCH = "v0.13.0"
ROOT_DIR = "s3://us-west-2.opendata.source.coop/carbonplan/srm-downscaling/output/production/"
STORE_SUBSET_ID = "global"

In [ ]:
def store_uri(gcm: str, root_dir: str = ROOT_DIR) -> str:
    return f"{root_dir}{gcm}-ERA5-{STORE_SUBSET_ID}.icechunk"

In [ ]:
trees: dict[str, xr.DataTree] = {}
repos: dict[str, icechunk.Repository] = {}
open_errors: dict[str, str] = {}

for gcm in GCMS:
    try:
        # region + from_env are required for writes; _icechunk_storage_for_path omits both
        _bucket, _, _prefix = store_uri(gcm).removeprefix("s3://").partition("/")
        repo = icechunk.Repository.open(
            icechunk.s3_storage(bucket=_bucket, prefix=_prefix, region="us-west-2", from_env=True)
        )
        repos[gcm] = repo
        session = repo.readonly_session(BRANCH) if BRANCH else repo.readonly_session()
        trees[gcm] = xr.open_datatree(session.store, engine="zarr", chunks={})
    except Exception as exc:  # noqa: BLE001  # report every failure, do not stop at the first
        open_errors[gcm] = f"{type(exc).__name__}: {exc}"

for gcm, err in open_errors.items():
    print(f"FAILED to open {gcm}: {err}")

# GCMs whose store actually opened. A GCM whose run is still in flight is skipped here rather
# than failing the whole notebook, so this check can be run against a partially-landed run.
OPEN_GCMS = tuple(gcm for gcm in GCMS if gcm in trees)
print(f"opened {len(OPEN_GCMS)}/{len(GCMS)} stores on branch {BRANCH!r}: {', '.join(OPEN_GCMS)}")

trees[OPEN_GCMS[0]]

In [ ]:
gcms, scenarios, variables, ensembles = [], [], [], []

for gcm in OPEN_GCMS:
    tree = trees[gcm]
    for scenario_name, scenario_node in tree.children.items():
        for var_name, var_node in scenario_node.children.items():
            for ens_name, ens_node in var_node.children.items():
                gcms.append(gcm)
                scenarios.append(scenario_name)
                variables.append(var_name)
                ensembles.append(ens_name)

print(f"{len(gcms)} leaves across {len(OPEN_GCMS)} GCMs")

In [ ]:
keep_idx = [i for i, s in enumerate(scenarios) if s != "debiased_coarse"]
gcms = [gcms[i] for i in keep_idx]
scenarios = [scenarios[i] for i in keep_idx]
variables = [variables[i] for i in keep_idx]
ensembles = [ensembles[i] for i in keep_idx]

keep_idx = [i for i, v in enumerate(variables) if v != "dtr"]
gcms = [gcms[i] for i in keep_idx]
scenarios = [scenarios[i] for i in keep_idx]
variables = [variables[i] for i in keep_idx]
ensembles = [ensembles[i] for i in keep_idx]

keep_idx = [i for i, v in enumerate(variables) if v != "hurs"]
gcms = [gcms[i] for i in keep_idx]
scenarios = [scenarios[i] for i in keep_idx]
variables = [variables[i] for i in keep_idx]
ensembles = [ensembles[i] for i in keep_idx]

In [ ]:
tags = []
for i, gcm in enumerate(gcms):
    var = variables[i]
    scenario = scenarios[i]
    ens = ensembles[i]
    tag = f"{gcm}_{var}_{scenario}_{ens}"
    tags.append(tag)

In [ ]:
gcms_np = np.array(gcms)
scenarios_np = np.array(scenarios)
variables_np = np.array(variables)
ensembles_np = np.array(ensembles)
tags_np = np.array(tags)

In [ ]:
INTERMEDIATE_FLAG_DIR = "s3://carbonplan-scratch/srm/qaqc/flags/" + STORE_SUBSET_ID + "/"

# Get dataset

In [ ]:
tag = "CESM2-WACCM_rsds_ssp245_003"

[overall_flag_time_varying, overall_flag_time_invariant] = combine_intermediate_flags(
    tag=tag,
    flag_dir=INTERMEDIATE_FLAG_DIR,
    flag_list_time_varying=FLAG_LIST_TIME_VARYING,
    flag_list_time_invariant=FLAG_LIST_TIME_INVARIANT,
)

lat = overall_flag_time_invariant.lat
lon = overall_flag_time_invariant.lon

In [ ]:
print(len(tags))
# This loop takes about 15 minutes to run on v0.13.0 (31 global data arrays)
OVERWRITE = True

for i, tag in tqdm(enumerate(tags)):
    print(tag)
    [gcm, var, scenario, ens] = parse_tag(tag)
    print("calculating flags")
    [overall_flag_time_varying, overall_flag_time_invariant] = combine_intermediate_flags(
        tag=tag,
        flag_dir=INTERMEDIATE_FLAG_DIR,
        flag_list_time_varying=FLAG_LIST_TIME_VARYING,
        flag_list_time_invariant=FLAG_LIST_TIME_INVARIANT,
    )

    group = f"{scenario}/{var}/{ens}"
    session = repos[gcm].writable_session(BRANCH)

    print("  writing flag_time_varying")
    write_final_qa_flags(
        session=session,
        group=group,
        flag_data=overall_flag_time_varying,
        flag_name="flag_time_varying",
        attrs=ATTRS_TIME_VARYING,
        overwrite=OVERWRITE,
    )

    print("  writing flag_time_invariant")
    write_final_qa_flags(
        session=session,
        group=group,
        flag_data=overall_flag_time_invariant,
        flag_name="flag_time_invariant",
        attrs=ATTRS_TIME_INVARIANT,
        overwrite=OVERWRITE,
    )
    commit = session.commit(f"write qa flags for {tag}")
    print(f"    commit {commit}")

# Evaluate flag prevalence

In [ ]:
import rusterize as _rusterize_pkg
from rasterix.rasterize import geometry_mask

from srm import catalog

ocean_mask_geoparquet = catalog.get("ocean-mask")

In [ ]:
# Claude-generated patch to get the geoparquet -> gridded mask to work
_original_rusterize = _rusterize_pkg.rusterize


def _rusterize_patched(*args, **kwargs):
    if kwargs.get("res") is not None and kwargs.get("out_shape") is not None:
        kwargs = dict(kwargs)
        kwargs.pop("res")
    return _original_rusterize(*args, **kwargs)


_rusterize_pkg.rusterize = _rusterize_patched

In [ ]:
tag = tags[0]
example_flags = get_intermediate_flags(flag_dir=INTERMEDIATE_FLAG_DIR, tag=tag)
target_grid_da = example_flags["flag_time_invariant"]

# rusterize requires lat sorted descending; reorder target_grid_da first if needed
template = target_grid_da.sortby("lat", ascending=False).proj.assign_crs(spatial_ref="epsg:4326")

land_mask = ~geometry_mask(
    template,
    ocean_mask_geoparquet.to_geodataframe()[["geom"]],
    all_touched=True,
    engine="rusterize",
    xdim="lon",
    ydim="lat",
).drop_vars("spatial_ref", errors="ignore")

land_mask.load()

land_mask_u8 = land_mask.astype(np.uint8)  # 0/1, still 1 byte
n_land_time_invariant = int(land_mask_u8.sum())  # land pixel count, compute once

# time-varying: land count is per-timestep-constant since land_mask has no time dim
n_land = n_land_time_invariant

In [ ]:
land_mask_u8.plot()

In [ ]:
# Note none of these fractions are area-weighted
SUMMARY_CSV = "qa_flag_summary.csv"

# land_mask is otherwise recomputed (full coastline rasterization) on every .where() use below
land_mask = land_mask.compute()

summary_rows = []

for i, tag in enumerate(tags):
    [gcm, var, scenario, ens] = parse_tag(tag)

    flags = get_intermediate_flags(flag_dir=INTERMEDIATE_FLAG_DIR, tag=tag)
    flag_time_varying = flags["flag_time_varying"]
    flag_time_invariant = flags["flag_time_invariant"]
    flag_time_varying_land = flag_time_varying * land_mask_u8
    flag_time_invariant_land = flag_time_invariant * land_mask_u8
    n_time = flag_time_varying.sizes["time"]

    # We calculate the land_frac means with sum() divided by n_land instead of
    # mean() with a mask so it goes quickly with the uint8 data type
    (
        frac_flagged_time_varying,
        frac_space_flagged_time_varying,
        frac_space_flagged_time_invariant,
        land_frac_flagged_time_varying,
        land_frac_space_flagged_time_varying,
        land_frac_space_flagged_time_invariant,
    ) = dask.compute(
        np.nanmean(flag_time_varying),
        np.nanmean(flag_time_varying.sum(dim="time") > 0),
        np.nanmean(flag_time_invariant),
        flag_time_varying_land.sum() / (n_land * n_time),
        (flag_time_varying_land.sum(dim="time") > 0).sum() / n_land,
        flag_time_invariant_land.sum() / n_land,
    )

    # Sanity check: these should each be 0 or 1
    # print(f"  check flag_time_varying max/min:   {np.nanmax(flag_time_varying)} / {np.nanmin(flag_time_varying)}")
    # print(f"  check flag_time_invariant max/min: {np.nanmax(flag_time_invariant)} / {np.nanmin(flag_time_invariant)}")

    print(f"{tag}:")
    print(
        f"  frac of dataset [time, lat, lon] flagged (time-varying):     {frac_flagged_time_varying:.4g}"
    )
    print(
        f"  frac of space [lat, lon] with >=1 flagged timestep:          {frac_space_flagged_time_varying:.4g}"
    )
    print(
        f"  frac of space [lat, lon] with time-invariant flag:           {frac_space_flagged_time_invariant:.4g}"
    )
    print(
        f"  land frac of dataset [time, lat, lon] flagged (time-varying):{land_frac_flagged_time_varying:.4g}"
    )
    print(
        f"  land frac of space [lat, lon] with >=1 flagged timestep:     {land_frac_space_flagged_time_varying:.4g}"
    )
    print(
        f"  land frac of space [lat, lon] with time-invariant flag:      {land_frac_space_flagged_time_invariant:.4g}"
    )

    summary_rows.append(
        {
            "tag": tag,
            "gcm": gcm,
            "variable": var,
            "scenario": scenario,
            "ensemble_member": ens,
            "frac_flagged_time_varying": float(frac_flagged_time_varying),
            "frac_space_flagged_time_varying": float(frac_space_flagged_time_varying),
            "frac_space_flagged_time_invariant": float(frac_space_flagged_time_invariant),
            "land_frac_flagged_time_varying": float(land_frac_flagged_time_varying),
            "land_frac_space_flagged_time_varying": float(land_frac_space_flagged_time_varying),
            "land_frac_space_flagged_time_invariant": float(land_frac_space_flagged_time_invariant),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)
summary_df

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

# reuse summary_df directly if it's already in memory from the cell above,
# otherwise: summary_df = pd.read_csv(SUMMARY_CSV)

metrics = [
    ("frac_flagged_time_varying", "time varying: frac flagged total"),
    ("land_frac_flagged_time_varying", "time varying: land frac flagged total"),
    ("frac_space_flagged_time_varying", "time varying: frac space flagged for >=1 timestep"),
    (
        "land_frac_space_flagged_time_varying",
        "time varying: land frac space flagged for >=1 timestep",
    ),
    ("frac_space_flagged_time_invariant", "time-invariant: frac space flagged"),
    ("land_frac_space_flagged_time_invariant", "time-invariant: land frac space flagged"),
]

variables = summary_df["variable"].unique().tolist()
scenarios = summary_df["scenario"].unique().tolist()

var_pos = {v: i for i, v in enumerate(variables)}
palette = dict(zip(scenarios, sns.color_palette(n_colors=len(scenarios))))
palette["historical"] = "black"

GROUP_SPREAD = 0.15  # distance between scenario clusters within a variable's x position
JITTER_WIDTH = 0.04  # random scatter within each scenario's own cluster

scenario_offset = {
    s: (i - (len(scenarios) - 1) / 2) * GROUP_SPREAD for i, s in enumerate(scenarios)
}

rng = np.random.default_rng(0)

fig, axes = plt.subplots(3, 2, figsize=(11, 12), sharey=True)
axes = axes.flatten()

for ax, (col, label) in zip(axes, metrics):
    for s in scenarios:
        sub = summary_df[summary_df["scenario"] == s]
        x = (
            sub["variable"].map(var_pos)
            + scenario_offset[s]
            + rng.uniform(-JITTER_WIDTH, JITTER_WIDTH, len(sub))
        )
        ax.scatter(
            x,
            sub[col],
            marker="o",
            facecolors="none",
            edgecolors=palette[s],
            linewidths=1.5,
            s=80,
            label=s,
        )
    ax.set_xticks(range(len(variables)))
    ax.set_xticklabels(variables, rotation=45)
    ax.set_title(label)
    ax.set_xlabel("")
    ax.set_ylabel("fraction")
    ax.grid()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="scenario",
    loc="upper center",
    bbox_to_anchor=(0.5, 1.04),
    ncol=len(scenarios),
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# Zoom in on one example

In [ ]:
tag = "CESM2-WACCM_pr_g6_1p5k_002"
flags = get_intermediate_flags(flag_dir=INTERMEDIATE_FLAG_DIR, tag=tag)
flag_time_varying = flags["flag_time_varying"]
flag_time_invariant = flags["flag_time_invariant"]
flag_time_varying_land = flag_time_varying * land_mask_u8
flag_time_invariant_land = flag_time_invariant * land_mask_u8
n_time = flag_time_varying.sizes["time"]

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(8, 5))
ax = plt.axes(projection=ccrs.PlateCarree())
flag_time_invariant.plot(ax=ax, transform=ccrs.PlateCarree(), cmap=plt.cm.viridis)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="k")
ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="k")
plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 5))
ax = plt.axes(projection=ccrs.PlateCarree())
freq_flagged = flag_time_varying_land.mean(dim="time")
freq_flagged.where(freq_flagged > 0).plot(ax=ax, transform=ccrs.PlateCarree(), cmap=plt.cm.viridis)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="0.2")
ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="0.8")
plt.show()

In [ ]:
(flag_time_varying_land.sum(dim=["lat", "lon"]).rolling(time=365).sum() / n_land).plot()
plt.ylabel("Fraction of land grid cells flagged in 365-day window")